# What Citizens Talk About — analysis notebook

**Lexical entity detection and negative review burden in online reviews of a public healthcare system (Andalusia, Spain).**

This notebook reproduces every number, table, figure and multimedia appendix of the paper from the raw review file. The lexicon (original and final) lives in `paper2_lexicon.py`; everything else is here.

| Section | Content | Needs |
|---|---|---|
| 0 | Setup and configuration | — |
| 1 | Canonical corpus (47,589 reviews / 812 facilities) | `data/Merged.csv` |
| 2 | Entity and marker detection with the final lexicon | corpus |
| 3 | Pre-validation lexical audit (origin of the disambiguation rules) | corpus |
| 4 | Validation sample (Frames A and B), checked against the annotated sample | corpus |
| 5 | Validation metrics: precision, recall, F1, Cohen κ | **restricted** annotation files |
| 6 | Sensitivity to exact duplicate records | corpus |
| 7 | Table 1, Figure 1, Figure 2 | corpus |
| 8 | Table 2 and the n/N dossier | corpus |
| 9 | Multimedia Appendices 1–4 | corpus |
| 10 | Table 3 and Multimedia Appendix 5 | output of section 5 |

**Data.** Neither the review corpus nor the annotation workbooks are distributed (see `README.md`). Sections 5 and 10 are skipped automatically when the restricted files are absent.

**Workflow fixed in advance.** Final lexicon fixed before manual validation → sample → blind annotation → metrics → results. The lexicon was not subsequently modified: low precision is reported, not corrected.

## 0 · Setup

In [ ]:
from __future__ import annotations

import hashlib
import platform
import re
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import openpyxl
import pandas as pd
import sklearn
import statsmodels
import statsmodels.formula.api as smf
from matplotlib.lines import Line2D
from sklearn.metrics import cohen_kappa_score

from paper2_lexicon import (CON_ORIG, CON_V2, ENT_ORIG, ENT_V2_REGEX, ENT_V2_SUBSTR,
                            INFO_ANCLAS, CITA_ESTRICTA, _info_con_contexto,
                            detectar_constructos, detectar_entidades)

# ── configuration ──────────────────────────────────────────────────────────
DATA_PATH = Path("data/Merged.csv")      # raw review file (not distributed)
RESTRICTED = Path("restricted")          # annotation workbooks (not distributed)
OUT = Path("results")                    # everything this notebook writes
SEED = 42                                # seed of the annotated validation sample
N_BOOT = 2000                            # bootstrap replicates
WRITE_ANNOTATION_WORKBOOK = False        # True only to regenerate the blind workbook

# restricted inputs
LIBRO_RESULTADO = RESTRICTED / "validacion_manual_paper2_resultado.xlsx"  # R1 + R2 annotation
LIBRO_METADATA = RESTRICTED / "validacion_manual_paper2_METADATA.xlsx"    # strata and weights
LIBRO_ADJUDICACION = RESTRICTED / "adjudicacion_pendiente.xlsx"           # cell-level adjudication

DIRS = {k: OUT / k for k in ["validation", "sensitivity", "tables", "figures",
                             "multimedia_appendices"]}
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
plt.rcParams.update({"font.size": 9, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25, "figure.dpi": 150})

print("Python", platform.python_version())
for m in (pd, np, statsmodels, sklearn, matplotlib, openpyxl):
    print(f"{m.__name__:<12}{m.__version__}")

In [ ]:
# ── labels and fixed design constants ─────────────────────────────────────
ETIQ_EN = {"Sistema/Gestión": "System/Management", "Médico": "Medical staff",
           "Administrativos": "Administrative staff", "Enfermería": "Nursing",
           "Urgencias": "Emergency services", "Instalaciones": "Facilities"}
ETIQ_M = {"pat_acceso": "Access barriers", "pat_espera": "Waiting and delay",
          "pat_evaluativo": "Evaluative judgement", "pat_sistema": "Systemic criticism",
          "pat_gratitud": "Gratitude", "pat_profesionalidad": "Professionalism",
          "pat_humanizacion": "Humanistic care"}
MARCADORES_POSITIVOS = ("pat_gratitud", "pat_profesionalidad", "pat_humanizacion")

# Facilities failed validation: reported as exploratory, never interpreted.
ENTIDADES_FIABLES = ["Sistema/Gestión", "Médico", "Administrativos", "Enfermería", "Urgencias"]

# Registry facility type -> analytic category (nine categories; the registry spells
# the health-centre category in two ways).
MAPA_TIPO = {
    "Centro de consultas externas": "Outpatient consultation centre",
    "Centro de Salud": "Health centre", "Centro de salud": "Health centre",
    "Consultorio": "Local health post",
    "Hospital concertado": "Contracted hospital",
    "Centro de hospitalización": "General hospital",
    "Hospital comarcal": "District hospital",
    "Hospital de alta resolución": "High-resolution hospital",
    "Hospital regional": "Regional hospital",
    "Hospital de especialidades": "Specialty hospital",
}

COL = {"Primaria": "#C0392B", "Hospitalaria": "#2C6E9B", "gris": "#555555"}
K_OR = "C(Atencion, Treatment('Hospitalaria'))[T.Primaria]"   # primary vs hospital


def formatear(path, anchos=None, wrap_cols=(), freeze="A2"):
    """Publication formatting for xlsx appendices: widths, wrap, bold pinned header row."""
    from openpyxl.styles import Alignment, Font
    from openpyxl.utils import get_column_letter
    wb = openpyxl.load_workbook(path)
    for ws in wb.worksheets:
        ws.freeze_panes = freeze
        cols = [c.value for c in ws[1]]
        for j, nombre in enumerate(cols, start=1):
            largo = max([len(str(nombre))] + [len(str(ws.cell(i, j).value or ""))
                                              for i in range(2, min(ws.max_row, 400) + 1)])
            ws.column_dimensions[get_column_letter(j)].width = \
                (anchos or {}).get(nombre, min(max(largo + 3, 12), 62))
            if nombre in wrap_cols:
                for i in range(2, ws.max_row + 1):
                    ws.cell(i, j).alignment = Alignment(wrap_text=True, vertical="top")
        for c in ws[1]:
            c.font = Font(bold=True)
            c.alignment = Alignment(wrap_text=True, vertical="center")
        ws.row_dimensions[1].height = 30
    wb.save(path)


def logit_or(df, y, cluster="Nombre"):
    """OR primary vs hospital for a binary outcome, SE clustered by facility."""
    m = smf.logit(f"{y} ~ C(Atencion, Treatment('Hospitalaria'))", data=df).fit(
        disp=0, cov_type="cluster", cov_kwds={"groups": df[cluster]})
    ci = np.exp(m.conf_int())
    return float(np.exp(m.params[K_OR])), float(ci.loc[K_OR, 0]), float(ci.loc[K_OR, 1]), \
        float(m.pvalues[K_OR])

## 1 · Canonical corpus

Filtering chain inherited unchanged from the longitudinal study (Paper 1): valid star rating (1–5) → non-empty text → ≥5 words in the raw text → removal of English-language reviews (share of 10 English function words > 0.2). Star rating defines review-level sentiment: ≤2 negative, 3 neutral, ≥4 positive. *Especializada* facilities are merged into hospital care.

The reviewer name column is dropped on load (data minimisation). `review_id` is a content hash (facility | date | stars | text), stable against row reordering; exact duplicates receive an occurrence suffix.

In [ ]:
def limpiar_espanol(text: str) -> str:
    text = str(text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#\w+", "", text)
    text = re.sub(r"http\S+|www\.\S+", "", text)
    text = re.sub(r"[^a-zA-ZáéíóúÁÉÍÓÚñÑüÜ\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip().lower()


def cargar_corpus(path=DATA_PATH) -> pd.DataFrame:
    """Rebuild the canonical corpus (47,589 reviews / 812 facilities) and verify it."""
    data = pd.read_csv(path, sep=";", low_memory=False)
    data = data.drop(columns=["name"], errors="ignore")       # reviewer names never used
    assert "name" not in data.columns

    data["stars"] = pd.to_numeric(data["stars"], errors="coerce")
    data = data[data["stars"].between(1, 5)]
    data["Fecha"] = pd.to_datetime(data["Fecha"], errors="coerce")
    data["year"] = data["Fecha"].dt.year
    for col in ["Atencion", "Ciudad", "Municipio", "Tipo_de_centro", "Dependencia", "Nombre"]:
        if col in data.columns:
            data[col] = data[col].astype("string").str.strip()
    data["Atencion"] = data["Atencion"].replace("Especializada", "Hospitalaria")
    data["negativa"] = (data["stars"] <= 2).astype(int)
    data["sentimiento"] = np.select(
        [data["stars"] <= 2, data["stars"] == 3], ["negativo", "neutro"], default="positivo")

    con_texto = data[data["text"].notna() & (data["text"].astype(str).str.strip() != "")].copy()
    con_texto["length_words"] = con_texto["text"].fillna("").apply(lambda x: len(str(x).split()))
    corpus = con_texto[con_texto["length_words"] >= 5].copy()

    corpus["text_clean"] = corpus["text"].apply(limpiar_espanol)
    stop_en = {"the", "and", "was", "were", "is", "are", "to", "of", "in", "on"}

    def es_ingles(t: str) -> bool:
        toks = t.split()
        return (sum(1 for x in toks if x in stop_en) / len(toks) > 0.2) if toks else False

    corpus = corpus[~corpus["text_clean"].apply(es_ingles)].reset_index(drop=True)
    assert len(corpus) == 47589, f"corpus = {len(corpus)}, expected 47,589"
    assert corpus["Nombre"].nunique() == 812, f"facilities = {corpus['Nombre'].nunique()}, expected 812"
    return corpus


def clave_contenido(df: pd.DataFrame) -> pd.Series:
    return (df["Nombre"].astype(str) + "|" + df["Fecha"].astype(str) + "|"
            + df["stars"].astype(str) + "|" + df["text"].astype(str))


def asignar_review_id(corpus: pd.DataFrame) -> pd.DataFrame:
    base = clave_contenido(corpus).apply(lambda s: hashlib.sha1(s.encode("utf-8")).hexdigest()[:12])
    ocurrencia = base.groupby(base).cumcount()
    corpus = corpus.copy()
    corpus["review_id"] = np.where(ocurrencia == 0, base, base + "-" + ocurrencia.astype(str))
    assert corpus["review_id"].is_unique
    return corpus


corpus = asignar_review_id(cargar_corpus())
corpus["tipo"] = corpus["Tipo_de_centro"].map(MAPA_TIPO)
DUP = clave_contenido(corpus).duplicated(keep="first").to_numpy()

N = len(corpus)
print(f"corpus: {N:,} reviews / {corpus['Nombre'].nunique()} facilities")
print(corpus["Atencion"].value_counts().to_string())
print(f"date range: {corpus['Fecha'].min().date()} – {corpus['Fecha'].max().date()}")
print(f"exact duplicate records (second copies): {DUP.sum():,} ({DUP.mean()*100:.1f}%)")

## 2 · Detection with the final lexicon

Entities are detected by substring/regex matching on the cleaned text; the seven discourse markers by phrase matching. **Sentiment is review-level** (star rating): an entity mention inherits the valence of the review, not of the aspect. The original lexicon is kept only for sensitivity analysis S2. (Internally, `"v2"` / `*_V2` denote the final lexicon and `"orig"` / `*_ORIG` the original one.)

In [ ]:
T = corpus["text_clean"]
ENT = {k: np.asarray(v, bool) for k, v in detectar_entidades(T, "v2").items()}
CON = {k: np.asarray(v, bool) for k, v in detectar_constructos(T, "v2").items()}
ENT_O = {k: np.asarray(v, bool) for k, v in detectar_entidades(T, "orig").items()}
CON_O = {k: np.asarray(v, bool) for k, v in detectar_constructos(T, "orig").items()}

NEG = corpus["negativa"].to_numpy().astype(bool)
NIV = corpus["Atencion"].astype(str).to_numpy()
CEN = corpus["Nombre"].astype(str).to_numpy()
PRIM, HOSP = NIV == "Primaria", NIV == "Hospitalaria"

cob = np.logical_or.reduce(list(ENT.values()))
print(f"coverage (>=1 entity): {cob.mean()*100:.1f}% ({cob.sum():,}/{N:,})")
pd.DataFrame({ETIQ_EN[e]: {"mentions": int(m.sum()), "prevalence %": m.mean()*100,
                           "% negative": NEG[m].mean()*100, "burden %": (m & NEG).mean()*100}
              for e, m in ENT.items()}).T.round(1)

## 3 · Pre-validation lexical audit

Before validation, every trigger of the **original** lexicon was audited for substring over-matching. `n_solo` counts reviews in which the trigger is the *only* one responsible for the entity/marker: if the trigger is spurious, those reviews are pure false positives. This audit is the origin of the disambiguation rules of the final lexicon, fixed before manual validation (listed in the docstring of `paper2_lexicon.py`); the comparison below quantifies their impact. Example review texts are deliberately not printed.

In [ ]:
def riesgo_por_trigger(lex: dict) -> pd.DataFrame:
    filas = []
    for grupo, terms in lex.items():
        hits = {t: T.str.contains(re.escape(t), regex=True, na=False).to_numpy() for t in terms}
        total = np.logical_or.reduce(list(hits.values()))
        for t, m in hits.items():
            otros = (np.logical_or.reduce([v for k, v in hits.items() if k != t])
                     if len(terms) > 1 else np.zeros(len(T), bool))
            solo = m & ~otros
            filas.append({"group": grupo, "trigger": t, "n": int(m.sum()),
                          "n_solo": int(solo.sum()),
                          "pct_of_group": round(solo.sum() / total.sum() * 100, 1)})
    return pd.DataFrame(filas).sort_values(["group", "n_solo"], ascending=[True, False])


audit_ent = riesgo_por_trigger(ENT_ORIG)
audit_con = riesgo_por_trigger(CON_ORIG)
audit_ent.groupby("group").head(4)

In [ ]:
# Specific over-matching hypotheses: share of single-trigger reviews matching the suspect form
casos = [
    ("cita", r"felicit", "Sistema/Gestión", "felicitar/felicito contain 'cita'"),
    ("espera", r"esperanza", "Sistema/Gestión", "'esperanza' contains 'espera'"),
    ("due", r"duele|dolor de|dueñ|duerm|duda", "Enfermería", "duele/dueño/duermen contain 'due'"),
    ("lista", r"especialista", "Sistema/Gestión", "'especialista' contains 'lista'"),
    ("sala", r"sala de espera|sala espera", "Instalaciones", "'sala de espera' denotes waiting"),
    ("información", r"informaci", "Administrativos", "generic 'información'"),
    ("auxiliar", r"auxiliar", "Enfermería", "'auxiliar' without nursing context"),
]
filas = []
for trig, patron, grupo, nota in casos:
    m_trig = T.str.contains(re.escape(trig), regex=True, na=False)
    otros = np.logical_or.reduce([T.str.contains(re.escape(t), regex=True, na=False)
                                  for t in ENT_ORIG[grupo] if t != trig])
    solo = m_trig & ~otros
    sosp = solo & T.str.contains(patron, regex=True, na=False)
    filas.append({"trigger": trig, "entity": grupo, "n_solo": int(solo.sum()),
                  "n_suspect": int(sosp.sum()),
                  "pct_suspect": round(sosp.sum() / solo.sum() * 100, 1) if solo.sum() else 0.0,
                  "hypothesis": nota})
pd.DataFrame(filas)

In [ ]:
# Original vs final lexicon: entity counts, negativity and burden
imp = pd.DataFrame([{"entity": ETIQ_EN[e],
                     "n_original": int(ENT_O[e].sum()), "n_final": int(ENT[e].sum()),
                     "pct_neg_original": NEG[ENT_O[e]].mean()*100, "pct_neg_final": NEG[ENT[e]].mean()*100,
                     "burden_original": (ENT_O[e] & NEG).mean()*100, "burden_final": (ENT[e] & NEG).mean()*100}
                    for e in ENT]).round(1)
print(f"coverage original lexicon {np.logical_or.reduce(list(ENT_O.values())).mean()*100:.1f}% -> "
      f"final lexicon {np.logical_or.reduce(list(ENT.values())).mean()*100:.1f}%")

# 'cita': initial anchor (adopted) vs strict \bcitas?\b, which would lose legitimate forms
m_sub = T.str.contains("cita", regex=False, na=False)
m_ancla = T.str.contains(r"\bcita", regex=True, na=False)
m_estr = T.str.contains(CITA_ESTRICTA, regex=True, na=False)
print(f"'cita' substring {m_sub.sum():,} · \\bcita {m_ancla.sum():,} · "
      f"\\bcitas?\\b {m_estr.sum():,} (would drop {int((m_ancla & ~m_estr).sum()):,} more)")
imp

## 4 · Validation sample

Two disjoint frames, n = 400:

* **Frame A** (n = 200): random, stratified by level of care (100 + 100), annotated blind → unbiased recall.
* **Frame B** (n = 200): targeted at risky triggers, prioritising reviews in which the trigger is the *only* cause of the flag (waiting-marker families, `auxiliar`, surviving `sala`, `información` with context, surviving `due`) plus marker controls → precision of the triggers with a disambiguation rule (and of `auxiliar`, left unchanged).

Strata are assigned to the **whole corpus** before sampling, by priority, so that every review has a defined selection probability `P(sel) = p_A + (1 − p_A)·p_B(stratum)`. 50 reviews of each frame went to the second annotator (κ). There is no stratum for `lista`: *lista de espera* contains *espera*, another System/Management trigger, so that stratum is empty by construction — and the rule therefore removes only the *especialista* false positives at zero recall cost.

The cell below regenerates the sample from the seed and **asserts that it is identical to the sample that was annotated** (hash of the sorted `review_id`s).

In [ ]:
N_MARCO_A_POR_NIVEL, N_MARCO_B = 100, 200
N_CONCORDANCIA_A = N_CONCORDANCIA_B = 50
HASH_R1, HASH_R2 = "15a6e2673a46914b", "5cea6a829864dcc0"     # annotated sample (7 Oct 2026)

VARS_ANOTACION = ["ent_medico", "ent_enfermeria", "ent_administrativos", "ent_sistema_gestion",
                  "ent_instalaciones", "ent_urgencias", "con_acceso", "con_espera",
                  "con_evaluativo", "con_sistemico", "con_gratitud", "con_profesionalidad",
                  "con_humanizacion"]

# Waiting-marker families. Order matters: each review goes to the FIRST family that matches,
# so 'tardanza', 'retraso' and 'demora' contain only reviews flagged by that expression alone.
ESPERA_FAMILIAS = {
    "espera_explicita": ["sala espera", "salas espera", "horas esperando", "horas espera",
                         "hora esperando", "tiempo espera", "tiempos espera", "lista espera",
                         "listas espera", "mucha espera", "larga espera", "largas esperas",
                         "demasiada espera", "esperando horas", "esperando hora",
                         "llevo esperando", "llevamos esperando", "llevo hora", "llevo horas",
                         "sigo esperando", "seguimos esperando", "estuve esperando",
                         "estuvimos esperando", "hicieron esperar", "hacen esperar",
                         "haciendo esperar", "esperar horas", "esperé horas", "esperar hora"],
    "tardanza": ["tardan", "tardaron", "tardó", "tarda mucho"],
    "retraso": ["retraso", "retrasos", "retrasa", "con retraso"],
    "demora": ["demora", "demoras", "demorado"],
}


def _solo_por(serie, patron, otros, regex):
    """True where `patron` matches and none of `otros` does (single trigger)."""
    m = serie.str.contains(patron, case=False, na=False, regex=regex).to_numpy()
    if otros:
        m &= ~serie.str.contains("|".join(re.escape(o) for o in otros),
                                 case=False, na=False, regex=True).to_numpy()
    return m


def construir_estratos_B():
    """[(stratum, mask, n_requested)] in PRIORITY order: waiting -> auxiliar -> disambiguated triggers -> controls."""
    especs, asignados = [], np.zeros(N, bool)
    for fam, n_fam in [("espera_explicita", 15), ("tardanza", 10), ("retraso", 10), ("demora", 10)]:
        m = T.str.contains("|".join(re.escape(f) for f in ESPERA_FAMILIAS[fam]),
                           case=False, na=False, regex=True).to_numpy()
        m = m & CON["pat_espera"] & ~asignados
        asignados |= m
        especs.append((f"B_pat_espera:{fam}", m, n_fam))

    otros_enf = [t for t in ENT_V2_SUBSTR["Enfermería"] if t != "auxiliar"]
    m_aux = _solo_por(T, "auxiliar", otros_enf, regex=False)
    m_aux &= ~T.str.contains(r"\bdue\b", case=False, na=False, regex=True).to_numpy()
    especs.append(("B_auxiliar_solo", m_aux & ENT["Enfermería"], 30))

    m_sala = _solo_por(T, r"\bsalas?\b(?!\s+(?:de\s+)?espera)", ENT_V2_SUBSTR["Instalaciones"], regex=True)
    especs.append(("B_sala_superviviente", m_sala & ENT["Instalaciones"], 25))

    m_info = T.apply(_info_con_contexto).to_numpy()
    m_info &= ~T.str.contains("|".join(re.escape(o) for o in ENT_V2_SUBSTR["Administrativos"]),
                              case=False, na=False, regex=True).to_numpy()
    especs.append(("B_informacion_con_contexto", m_info & ENT["Administrativos"], 20))

    m_due = _solo_por(T, r"\bdue\b", otros_enf + ["auxiliar"], regex=True)
    especs.append(("B_due_superviviente", m_due & ENT["Enfermería"], 15))

    especs.append(("B_ctrl_gratitud", CON["pat_gratitud"], 18))
    especs.append(("B_ctrl_profesionalidad", CON["pat_profesionalidad"], 18))
    especs.append(("B_ctrl_acceso", CON["pat_acceso"], 18))
    especs.append(("B_ctrl_resto_constructos",
                   CON["pat_evaluativo"] | CON["pat_sistema"] | CON["pat_humanizacion"], 11))
    return especs


def asignar_estrato_full():
    """Deterministic stratum for EVERY review (first matching stratum wins)."""
    especs = construir_estratos_B()
    estrato = np.full(N, "", dtype=object)
    for nombre, mascara, _ in especs:
        estrato[(estrato == "") & mascara] = nombre
    return estrato, especs


ESTRATO_FULL, ESPECS_B = asignar_estrato_full()

In [ ]:
def muestrear(seed):
    rng = np.random.default_rng(seed)
    filas, auditoria = [], []

    # Frame A: random, stratified by level
    p_A, en_A = np.zeros(N), np.zeros(N, bool)
    for nivel in ["Primaria", "Hospitalaria"]:
        cand = corpus.index[corpus["Atencion"] == nivel].to_numpy()
        n = min(N_MARCO_A_POR_NIVEL, len(cand))
        sel = rng.choice(cand, size=n, replace=False)
        p_A[cand] = n / len(cand)
        en_A[sel] = True
        filas += [{"idx": int(i), "sample_frame": "A", "sampling_stratum": f"A_{nivel.lower()}",
                   "stratum_population": len(cand), "n_selected": n} for i in sel]
        auditoria.append({"sample_frame": "A", "sampling_stratum": f"A_{nivel.lower()}",
                          "candidatos": len(cand), "solicitados": N_MARCO_A_POR_NIVEL,
                          "seleccionados": n, "deficit": N_MARCO_A_POR_NIVEL - n, "pase": 1})

    # Frame B: disjoint from A, mutually exclusive strata
    pobl = {nombre: np.where((ESTRATO_FULL == nombre) & ~en_A)[0] for nombre, _, _ in ESPECS_B}
    seleccion = {}
    for nombre, _, n_pedido in ESPECS_B:                       # pass 1: requested allocation
        n = min(n_pedido, len(pobl[nombre]))
        seleccion[nombre] = rng.choice(pobl[nombre], size=n, replace=False) if n else np.array([], int)
        auditoria.append({"sample_frame": "B", "sampling_stratum": nombre,
                          "candidatos": len(pobl[nombre]), "solicitados": n_pedido,
                          "seleccionados": n, "deficit": n_pedido - n, "pase": 1})

    deficit = N_MARCO_B - sum(len(v) for v in seleccion.values())
    if deficit > 0:                                            # pass 2: redistribute shortfall
        sobrantes = {n: len(pobl[n]) - len(seleccion[n]) for n, _, _ in ESPECS_B}
        sobrantes = {n: v for n, v in sobrantes.items() if v > 0}
        total = sum(sobrantes.values())
        if total:
            extra = {n: int(round(deficit * v / total)) for n, v in sobrantes.items()}
            for nombre in sorted(extra, key=lambda x: -extra[x]):
                if deficit <= 0:
                    break
                k = min(extra[nombre], sobrantes[nombre], deficit)
                if k <= 0:
                    continue
                resto = np.setdiff1d(pobl[nombre], seleccion[nombre], assume_unique=False)
                k = min(k, len(resto))
                if k <= 0:
                    continue
                seleccion[nombre] = np.concatenate([seleccion[nombre],
                                                    rng.choice(resto, size=k, replace=False)])
                deficit -= k
                auditoria.append({"sample_frame": "B", "sampling_stratum": nombre,
                                  "candidatos": len(pobl[nombre]), "solicitados": k,
                                  "seleccionados": k, "deficit": 0, "pase": 2})

    for nombre, _, _ in ESPECS_B:
        filas += [{"idx": int(i), "sample_frame": "B", "sampling_stratum": nombre,
                   "stratum_population": len(pobl[nombre]), "n_selected": len(seleccion[nombre])}
                  for i in seleccion[nombre]]
    muestra = pd.DataFrame(filas)
    assert muestra["idx"].is_unique

    # Design weights: one weight per review, whichever frame it entered through.
    # p_B is conditional on the realised Frame A; section 5 calibrates to census cell totals.
    p_B = {n: len(seleccion[n]) / len(pobl[n]) if len(pobl[n]) else 0.0 for n, _, _ in ESPECS_B}
    idx = muestra["idx"].to_numpy()
    muestra["estrato_asignado"] = [ESTRATO_FULL[i] or "(ninguno)" for i in idx]
    muestra["p_marco_A"] = p_A[idx]
    muestra["p_marco_B"] = [p_B.get(ESTRATO_FULL[i], 0.0) for i in idx]
    muestra["sampling_prob"] = muestra["p_marco_A"] + (1 - muestra["p_marco_A"]) * muestra["p_marco_B"]
    muestra["sampling_weight"] = 1 / muestra["sampling_prob"]

    muestra["en_evaluador2"] = 0                               # second-annotator subsample
    for marco, n_conc in [("A", N_CONCORDANCIA_A), ("B", N_CONCORDANCIA_B)]:
        pool = muestra.index[muestra["sample_frame"] == marco].to_numpy()
        muestra.loc[rng.choice(pool, size=min(n_conc, len(pool)), replace=False), "en_evaluador2"] = 1
    muestra["seed"] = seed
    muestra["orden_seleccion"] = np.arange(len(muestra))
    return muestra, pd.DataFrame(auditoria)


def hash_ids(ids) -> str:
    return hashlib.sha1("|".join(sorted(map(str, ids))).encode()).hexdigest()[:16]


muestra, auditoria = muestrear(SEED)
muestra["review_id"] = corpus.loc[muestra["idx"], "review_id"].to_numpy()
h1 = hash_ids(muestra["review_id"])
h2 = hash_ids(muestra.loc[muestra["en_evaluador2"] == 1, "review_id"])
assert (h1, h2) == (HASH_R1, HASH_R2), "the regenerated sample differs from the annotated one"
print(f"sample = annotated sample ✓  (R1 {h1}, R2 {h2})")
print(f"frame A {int((muestra.sample_frame == 'A').sum())} · frame B {int((muestra.sample_frame == 'B').sum())}"
      f" · sum of design weights {muestra['sampling_weight'].sum():,.0f} vs N {N:,}")
auditoria.groupby(["sample_frame", "sampling_stratum"], as_index=False).agg(
    candidates=("candidatos", "max"), selected=("seleccionados", "sum"))

In [ ]:
# Annotation guide (Multimedia Appendix 2), as delivered to the annotators.
CODEBOOK = [
    ("GUÍA DE ANOTACIÓN — Paper 2 (validación del léxico v2)", ""),
    ("", ""),
    ("Qué se anota",
     "Para cada reseña, marque 1 si el elemento está GENUINAMENTE presente en el texto y 0 si no. "
     "Deje la celda vacía solo si no puede decidir."),
    ("Qué NO se anota",
     "NO juzgue si la reseña es positiva o negativa. La valencia procede de la estrella y ya fue "
     "validada en el Paper 1. Aquí solo se juzga PRESENCIA de entidad o constructo."),
    ("No intente adivinar el sistema",
     "Las reseñas no vienen marcadas y el orden está aleatorizado a propósito. Anote lo que lee, "
     "no lo que crea que el programa habría detectado."),
    ("", ""),
    ("ENTIDADES (6)", ""),
    ("ent_medico", "Personal médico: médico, doctor, facultativo, especialista, pediatra, cirujano."),
    ("ent_enfermeria", "Personal de enfermería: enfermero/a, DUE, auxiliar de enfermería o de clínica."),
    ("ent_administrativos",
     "Personal administrativo: administrativo/a, mostrador, ventanilla, recepción, celador, centralita."),
    ("ent_sistema_gestion",
     "Procesos organizativos: cita, teléfono, espera, lista de espera, agenda, app, burocracia."),
    ("ent_instalaciones",
     "Entorno físico: sala (por su estado), habitación, limpieza, equipamiento, parking, cafetería."),
    ("ent_urgencias", "Servicio de urgencias o emergencias, triaje."),
    ("", ""),
    ("CONSTRUCTOS (7)", ""),
    ("con_acceso", "Dificultad para acceder: no cogen el teléfono, no dan cita, no se puede contactar."),
    ("con_espera", "Espera o demora: esperar, tiempo de espera, retraso, demora, tardanza en ser atendido."),
    ("con_evaluativo", "Juicio sumario del centro: el peor centro, una vergüenza, cada vez peor, malas formas."),
    ("con_sistemico", "Crítica dirigida al sistema, no al centro: la sanidad pública, la seguridad social."),
    ("con_gratitud", "Agradecimiento explícito: gracias, quiero agradecer."),
    ("con_profesionalidad", "Elogio de la competencia: buenos profesionales, buena atención, buen trato."),
    ("con_humanizacion", "Dimensión relacional y humana: trato humano, calidad humana."),
    ("", ""),
    ("CRITERIOS FIJADOS A PRIORI (no reinterpretar durante la anotación)", ""),
    ("PRINCIPIO GENERAL",
     "Anote el CONCEPTO, no la expresión. El patrón de referencia debe ser independiente del "
     "sistema que se valida: si marcase un constructo solo porque aparece la frase que el "
     "programa busca, la validación se volvería circular y no mediría nada."),
    ("«sala de espera» — regla semántica",
     "La expresión «sala de espera» NO implica por sí sola el constructo de espera. "
     "con_espera = 1 únicamente cuando el contexto expresa tiempo de espera, demora o retraso. "
     "Puede contar como ent_instalaciones si se comenta el estado físico del espacio. "
     "Ambas categorías pueden coexistir si el texto expresa los dos contenidos."),
    ("  ejemplo 1", "«Estuve dos horas en la sala de espera» → con_espera = 1"),
    ("  ejemplo 2", "«La sala de espera estaba llena y tardaron muchísimo» → con_espera = 1"),
    ("  ejemplo 3", "«La sala de espera estaba muy limpia» → con_espera = 0, ent_instalaciones = 1"),
    ("  ejemplo 4",
     "«La sala de espera es pequeña pero el médico excelente» → con_espera = 0, ent_instalaciones = 1"),
    ("«sala de espera» — entidad",
     "Para ent_instalaciones rige el criterio operativo anterior: solo si se comenta el espacio "
     "físico. Para ent_sistema_gestion, marque 1 si el texto se refiere al proceso organizativo "
     "(esperar, cola, cita, orden de atención), no por la simple aparición de la expresión."),
    ("«información»",
     "Cuenta como ent_administrativos solo si se refiere al servicio o mostrador de información. "
     "El acto genérico de informar («no me dieron información») NO cuenta."),
    ("«auxiliar»",
     "Cuenta como ent_enfermeria solo si el contexto indica auxiliar de enfermería o de clínica. "
     "Si es administrativo o indeterminado, no cuenta."),
    ("Presencia del constructo",
     "Marque 1 si la reseña expresa ese contenido, con independencia de la formulación concreta. "
     "No se limite a las frases del sistema: así se mide también lo que el léxico se deja fuera."),
    ("Menciones de pasada",
     "Una mención breve cuenta. «La enfermera fue amable» → ent_enfermeria = 1."),
    ("Negación",
     "La mención cuenta aunque esté negada. «No había ningún médico» → ent_medico = 1."),
]

# Blind annotation sheets: only review_id, text and empty fields, in randomised order.
base = corpus.loc[muestra["idx"].to_numpy(), ["review_id", "text"]].reset_index(drop=True)
base = pd.concat([base, muestra[["sample_frame", "en_evaluador2"]].reset_index(drop=True)], axis=1)


def hoja_anotacion(df):
    h = df[["review_id", "text"]].rename(columns={"text": "texto_resena"})
    for v in VARS_ANOTACION:
        h[v] = ""
    h["notas"] = ""
    return h


hoja_r1 = hoja_anotacion(base.sample(frac=1.0, random_state=SEED).reset_index(drop=True))
hoja_r2 = hoja_anotacion(base[base["en_evaluador2"] == 1]
                         .sample(frac=1.0, random_state=SEED + 7).reset_index(drop=True))

# Blinding checks: no lexicon output, stratum, rating or level in the sheets; order mixed.
prohibidas = (set(ENT) | set(CON) | {"sample_frame", "sampling_stratum", "sampling_weight",
                                     "sampling_prob", "stars", "negativa", "sentimiento",
                                     "Atencion", "Nombre", "text_clean"})
assert not prohibidas & (set(hoja_r1.columns) | set(hoja_r2.columns))
marco = hoja_r1[["review_id"]].merge(muestra[["review_id", "sample_frame"]], on="review_id")["sample_frame"]
cambios = int((marco != marco.shift()).sum())
assert cambios > 50
print(f"blinding ✓  · frame changes along the R1 sheet: {cambios}")

if WRITE_ANNOTATION_WORKBOOK:            # restricted output: contains review text
    from openpyxl.worksheet.datavalidation import DataValidation
    from openpyxl.utils import get_column_letter
    RESTRICTED.mkdir(exist_ok=True)
    adjud = pd.DataFrame({"review_id": hoja_r2["review_id"]})
    for v in VARS_ANOTACION:
        adjud[v] = ""
    adjud["resuelto_por"] = ""
    with pd.ExcelWriter(RESTRICTED / "validacion_manual_paper2.xlsx", engine="openpyxl") as xl:
        pd.DataFrame(CODEBOOK, columns=["campo", "definicion"]).to_excel(xl, "CODEBOOK", index=False)
        for nombre, h in [("ANOTACION_R1", hoja_r1), ("ANOTACION_R2", hoja_r2), ("ADJUDICACION", adjud)]:
            h.to_excel(xl, sheet_name=nombre, index=False)
            ws = xl.book[nombre]
            dv = DataValidation(type="list", formula1='"1,0"', allow_blank=True)
            ws.add_data_validation(dv)
            cols = [c.value for c in ws[1]]
            for v in VARS_ANOTACION:
                letra = get_column_letter(cols.index(v) + 1)
                dv.add(f"{letra}2:{letra}{len(h) + 1}")
    # Metadata in a SEPARATE file: the stratum reveals the lexicon's prediction.
    meta_out = muestra.drop(columns="idx").copy()
    for c in ["Nombre", "Atencion", "stars"]:
        meta_out[c] = corpus.loc[muestra["idx"], c].to_numpy()
    with pd.ExcelWriter(LIBRO_METADATA, engine="openpyxl") as xl:
        meta_out.to_excel(xl, sheet_name="SAMPLE_METADATA", index=False)
        auditoria.to_excel(xl, sheet_name="SAMPLE_AUDIT", index=False)
    print("annotation workbooks written to", RESTRICTED)

## 5 · Validation metrics *(restricted data)*

**Reference standard.** Annotator 1 on all 400 reviews; on the 100 double-annotated reviews, disagreements were adjudicated cell by cell. κ is computed on the two independent annotations **before** adjudication and measures agreement between the two human annotators, not between lexicon and humans.

**Three estimators**, reported side by side (Multimedia Appendix 5):

* **A crude** — Frame A unweighted, Wilson intervals. Auditable by hand, but *not* a corpus-level estimator (Frame A was sampled 50:50 by level).
* **A post-stratified** — Frame A reweighted to the true level composition; bootstrap within level.
* **A+B calibrated** (primary) — design weights calibrated to the census size of each post-stratification cell (Frame B strata + residual cell, a partition fixed before sampling): `w_cal(i) = w(i)·N_c / Σ_{j∈c} w(j)`. Stratified bootstrap that recalibrates in every replicate.

In [ ]:
VAR_A_LEXICO = {
    "ent_medico": ENT["Médico"], "ent_enfermeria": ENT["Enfermería"],
    "ent_administrativos": ENT["Administrativos"], "ent_sistema_gestion": ENT["Sistema/Gestión"],
    "ent_instalaciones": ENT["Instalaciones"], "ent_urgencias": ENT["Urgencias"],
    "con_acceso": CON["pat_acceso"], "con_espera": CON["pat_espera"],
    "con_evaluativo": CON["pat_evaluativo"], "con_sistemico": CON["pat_sistema"],
    "con_gratitud": CON["pat_gratitud"], "con_profesionalidad": CON["pat_profesionalidad"],
    "con_humanizacion": CON["pat_humanizacion"],
}
# Target claimed by each targeted Frame B stratum (trigger-level precision)
DIANA_ESTRATO = {
    "B_auxiliar_solo": "ent_enfermeria", "B_due_superviviente": "ent_enfermeria",
    "B_sala_superviviente": "ent_instalaciones", "B_informacion_con_contexto": "ent_administrativos",
    "B_pat_espera:espera_explicita": "con_espera", "B_pat_espera:tardanza": "con_espera",
    "B_pat_espera:retraso": "con_espera", "B_pat_espera:demora": "con_espera",
    "B_ctrl_acceso": "con_acceso", "B_ctrl_gratitud": "con_gratitud",
    "B_ctrl_profesionalidad": "con_profesionalidad",
}


def wilson(k, n, z=1.96):
    if n <= 0:
        return (float("nan"), float("nan"))
    p = k / n
    d = 1 + z**2 / n
    c = (p + z**2 / (2 * n)) / d
    h = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / d
    return (max(0.0, c - h), min(1.0, c + h))


def _pr(gold, pred, w):
    tp = float((w * (gold & pred)).sum()); fp = float((w * (~gold & pred)).sum())
    fn = float((w * (gold & ~pred)).sum())
    prec = tp / (tp + fp) if tp + fp > 0 else float("nan")
    rec = tp / (tp + fn) if tp + fn > 0 else float("nan")
    f1 = 2 * prec * rec / (prec + rec) if (prec == prec and rec == rec and prec + rec > 0) else float("nan")
    return {"TP": tp, "FP": fp, "FN": fn, "precision": prec, "recall": rec, "f1": f1}


def calibrar(w, celdas, n_pobl):
    w = w.astype(float).copy()
    for c in np.unique(celdas):
        m = celdas == c
        s = w[m].sum()
        if s > 0 and n_pobl.get(c, 0) > 0:
            w[m] *= n_pobl[c] / s
    return w


def bootstrap_pr(gold, pred, celdas, w_dis, n_pobl, n_rep=N_BOOT, seed=99):
    """Bootstrap stratified by cell; weights recalibrated in every replicate."""
    rng = np.random.default_rng(seed)
    idx_por_celda = {c: np.where(celdas == c)[0] for c in np.unique(celdas)}
    out = {"precision": [], "recall": [], "f1": []}
    for _ in range(n_rep):
        tomados = np.concatenate([rng.choice(ix, size=len(ix), replace=True)
                                  for ix in idx_por_celda.values() if len(ix)])
        w = calibrar(w_dis[tomados], celdas[tomados], n_pobl)
        r = _pr(gold[tomados], pred[tomados], w)
        for k in out:
            out[k].append(r[k])
    return {k: (float(np.nanpercentile(v, 2.5)), float(np.nanpercentile(v, 97.5)))
            for k, v in out.items()}


RESTRINGIDOS_OK = all(p.exists() for p in [LIBRO_RESULTADO, LIBRO_METADATA, LIBRO_ADJUDICACION])
print("restricted annotation files available:", RESTRINGIDOS_OK)

In [ ]:
if RESTRINGIDOS_OK:
    r1 = pd.read_excel(LIBRO_RESULTADO, sheet_name="ANOTACION_R1").set_index("review_id")
    r2 = pd.read_excel(LIBRO_RESULTADO, sheet_name="ANOTACION_R2").set_index("review_id")
    meta = pd.read_excel(LIBRO_METADATA, sheet_name="SAMPLE_METADATA").set_index("review_id")

    # ── integrity: annotated sample, binary domain, completeness ─────────────
    assert hash_ids(r1.index) == HASH_R1 and hash_ids(r2.index) == HASH_R2
    assert r1.index.is_unique and set(r2.index) <= set(r1.index) and set(meta.index) == set(r1.index)
    for nom, df in [("R1", r1), ("R2", r2)]:
        vals = df[VARS_ANOTACION].stack().dropna()
        assert vals.isin([0, 1, 0.0, 1.0, "0", "1"]).all(), f"{nom}: non-binary values"
        vacias = int((df[VARS_ANOTACION].apply(pd.to_numeric, errors="coerce").isna()
                      .sum(axis=1) == len(VARS_ANOTACION)).sum())
        assert vacias == 0, f"{nom}: {vacias} rows not annotated"
    print(f"integrity ✓  R1 {len(r1)} rows · R2 {len(r2)} rows")

    # ── reference standard: R1, overwritten cell by cell by the adjudication ──
    gold = r1[VARS_ANOTACION].apply(pd.to_numeric, errors="coerce").fillna(0).astype(int)
    dec = pd.read_excel(LIBRO_ADJUDICACION, sheet_name="ADJUDICAR")
    dec["DECISION"] = pd.to_numeric(dec["DECISION"], errors="coerce")
    resueltas = dec.dropna(subset=["DECISION"])
    for _, r in resueltas.iterrows():
        gold.loc[r["review_id"], r["variable"]] = int(r["DECISION"])
    print(f"adjudicated cells applied: {len(resueltas)} of {len(dec)}")

    # ── post-stratification cells and calibrated weights ──────────────────
    celda_full = np.array([e if e else "(sin estrato)" for e in ESTRATO_FULL], dtype=object)
    n_pobl = pd.Series(celda_full).value_counts().to_dict()
    pos = corpus.set_index("review_id").index.get_indexer(meta.index)
    celdas = celda_full[pos]
    gold = gold.loc[meta.index]
    w_dis = meta["sampling_weight"].to_numpy(float)
    w_cal = calibrar(w_dis, celdas, n_pobl)
    es_A = (meta["sample_frame"] == "A").to_numpy()

    # Frame A post-stratification by level (Frame A was sampled 50:50)
    niv = meta["Atencion"].astype(str).to_numpy()
    n_pobl_niv = corpus["Atencion"].astype(str).value_counts().to_dict()
    n_A_niv = pd.Series(niv[es_A]).value_counts().to_dict()
    w_ps = np.array([n_pobl_niv.get(x, 0) / n_A_niv.get(x, 1) for x in niv], float)
    print(f"calibrated Σw = {w_cal.sum():,.0f} (N = {N:,}) · Frame A level weights "
          f"{ {k: round(n_pobl_niv[k] / n_A_niv[k], 1) for k in n_A_niv} }")
else:
    print("Section 5 skipped: restricted annotation files not found in", RESTRICTED)

In [ ]:
if RESTRINGIDOS_OK:
    filas = []
    for var in VARS_ANOTACION:
        g = gold[var].to_numpy().astype(bool)
        p = VAR_A_LEXICO[var][pos]
        a = _pr(g[es_A], p[es_A], np.ones(es_A.sum()))                    # A crude
        tpA, fpA, fnA = a["TP"], a["FP"], a["FN"]
        ciPA, ciRA = wilson(tpA, tpA + fpA), wilson(tpA, tpA + fnA)
        aps = _pr(g[es_A], p[es_A], w_ps[es_A])                           # A post-stratified
        bps = bootstrap_pr(g[es_A], p[es_A], niv[es_A], w_ps[es_A], n_pobl_niv, seed=123)
        c = _pr(g, p, w_cal)                                              # A+B calibrated
        boot = bootstrap_pr(g, p, celdas, w_dis, n_pobl)
        filas.append({
            "diana": var, "n_marcadas_corpus": int(VAR_A_LEXICO[var].sum()),
            "A_n_pred_pos": int(tpA + fpA), "A_n_gold_pos": int(tpA + fnA), "A_tp": int(tpA),
            "AB_n_pred_pos": int(p.sum()), "AB_tp": int((g & p).sum()),
            "A_crudo_precision": tpA / (tpA + fpA) if tpA + fpA else np.nan,
            "A_crudo_prec_lo": ciPA[0], "A_crudo_prec_hi": ciPA[1],
            "A_crudo_recall": tpA / (tpA + fnA) if tpA + fnA else np.nan,
            "A_crudo_rec_lo": ciRA[0], "A_crudo_rec_hi": ciRA[1],
            "Aps_precision": aps["precision"], "Aps_prec_lo": bps["precision"][0],
            "Aps_prec_hi": bps["precision"][1], "Aps_recall": aps["recall"],
            "Aps_rec_lo": bps["recall"][0], "Aps_rec_hi": bps["recall"][1], "Aps_f1": aps["f1"],
            "AB_precision": c["precision"], "AB_prec_lo": boot["precision"][0],
            "AB_prec_hi": boot["precision"][1], "AB_recall": c["recall"],
            "AB_rec_lo": boot["recall"][0], "AB_rec_hi": boot["recall"][1],
            "AB_f1": c["f1"], "AB_f1_lo": boot["f1"][0], "AB_f1_hi": boot["f1"][1],
            "prev_gold_pond_pct": float((w_cal * g).sum() / w_cal.sum() * 100),
            "prev_lexico_pct": VAR_A_LEXICO[var].mean() * 100,
        })
    val = pd.DataFrame(filas)

    # trigger-level precision within targeted Frame B strata (unweighted, Wilson)
    filas_e = []
    for est, diana in DIANA_ESTRATO.items():
        sel = (celdas == est) & ~es_A
        if sel.sum():
            k, n = int(gold[diana].to_numpy().astype(bool)[sel].sum()), int(sel.sum())
            lo, hi = wilson(k, n)
            filas_e.append({"estrato": est, "diana": diana, "n_anotadas": n, "n_correctas": k,
                            "precision": k / n, "ci_lo": lo, "ci_hi": hi,
                            "poblacion_estrato": n_pobl.get(est, 0)})
    val_est = pd.DataFrame(filas_e).sort_values("precision")

    # inter-annotator agreement BEFORE adjudication
    comunes = r2.index.intersection(r1.index)
    A1 = r1.loc[comunes, VARS_ANOTACION].apply(pd.to_numeric, errors="coerce").fillna(0).astype(int)
    A2 = r2.loc[comunes, VARS_ANOTACION].apply(pd.to_numeric, errors="coerce").fillna(0).astype(int)
    val_k = pd.DataFrame([{"variable": v, "n": len(comunes),
                           "acuerdo_bruto": float((A1[v] == A2[v]).mean()),
                           "kappa": cohen_kappa_score(A1[v], A2[v]),
                           "prev_R1_pct": A1[v].mean() * 100, "prev_R2_pct": A2[v].mean() * 100,
                           "n_desacuerdos": int((A1[v] != A2[v]).sum())} for v in VARS_ANOTACION])
    kappa_global = cohen_kappa_score(A1.to_numpy().ravel(), A2.to_numpy().ravel())
    print(f"global κ {kappa_global:.3f} · raw agreement {(A1.to_numpy() == A2.to_numpy()).mean():.1%} "
          f"({int((A1.to_numpy() == A2.to_numpy()).sum()):,}/{A1.size:,})")

    val.to_csv(DIRS["validation"] / "validation_metrics.csv", index=False)
    val_est.to_csv(DIRS["validation"] / "validation_precision_by_stratum.csv", index=False)
    val_k.to_csv(DIRS["validation"] / "validation_interrater_agreement.csv", index=False)
    display(val_k.round(3))
    display(val[["diana", "n_marcadas_corpus", "AB_precision", "AB_prec_lo", "AB_prec_hi",
                 "AB_recall", "AB_rec_lo", "AB_rec_hi", "AB_f1"]].round(3))
    display(val_est.round(3))

## 6 · Sensitivity to exact duplicate records

1,790 records (3.8%) are exact second copies (same facility, date, rating and text; every group has size 2) — collection artefacts. They are kept in the main analysis to preserve the corpus as defined in Paper 1; here every estimate is recomputed after removing them. For odds ratios the check is whether the deduplicated estimate falls inside the main 95% CI, not an absolute cut-off.

In [ ]:
def estimaciones(d: pd.DataFrame, escenario: str):
    d = d.reset_index(drop=True)
    ent = {k: np.asarray(v, bool) for k, v in detectar_entidades(d["text_clean"], "v2").items()}
    con = {k: np.asarray(v, bool) for k, v in detectar_constructos(d["text_clean"], "v2").items()}
    neg = d["negativa"].to_numpy().astype(bool)
    niv = d["Atencion"].to_numpy()
    prev, tipo, ors = [], [], []
    for e in ENTIDADES_FIABLES + ["Instalaciones"]:
        m = ent[e]
        fila = {"escenario": escenario, "entidad": e, "n_menciones": int(m.sum()),
                "prevalencia": m.mean() * 100, "pct_negativo": neg[m].mean() * 100,
                "burden": (m & neg).mean() * 100}
        for lvl, et in [("Primaria", "prim"), ("Hospitalaria", "hosp")]:
            sel = niv == lvl
            fila[f"prev_{et}"] = m[sel].mean() * 100
            fila[f"burden_{et}"] = (m & neg)[sel].sum() / sel.sum() * 100
        prev.append(fila)
    for e in ENTIDADES_FIABLES:
        b = pd.Series(ent[e] & neg, index=d.index)
        for t, g in d.groupby("tipo"):
            tipo.append({"escenario": escenario, "tipo": t, "entidad": e,
                         "burden": b.loc[g.index].mean() * 100, "n": len(g)})
    dm = d[d["Atencion"].isin(["Primaria", "Hospitalaria"])].copy()
    dm["Atencion"], dm["Nombre"] = dm["Atencion"].astype(str), dm["Nombre"].astype(str)
    for c in CON_V2:
        estrato = "positivo" if c in MARCADORES_POSITIVOS else "negativo"
        dm["y"] = con[c][dm.index.to_numpy()].astype(int)
        sub = dm[dm["sentimiento"] == estrato]
        OR, lo, hi, p = logit_or(sub, "y")
        ors.append({"escenario": escenario, "marcador": c, "estrato": estrato,
                    "n_estrato": len(sub), "OR": OR, "ci_lo": lo, "ci_hi": hi, "p": p})
    return prev, tipo, ors


res = [estimaciones(corpus, "principal"), estimaciones(corpus[~DUP], "deduplicado")]
s_prev = pd.DataFrame(res[0][0] + res[1][0])
s_tipo = pd.DataFrame(res[0][1] + res[1][1])
s_or = pd.DataFrame(res[0][2] + res[1][2])
s_prev.to_csv(DIRS["sensitivity"] / "duplicates_prevalence_burden.csv", index=False)
s_tipo.to_csv(DIRS["sensitivity"] / "duplicates_burden_by_facility_type.csv", index=False)
s_or.to_csv(DIRS["sensitivity"] / "duplicates_odds_ratios.csv", index=False)


def max_dif(df, idx, val):
    p = df.pivot_table(index=idx, columns="escenario", values=val)
    return float((p["deduplicado"] - p["principal"]).abs().max())


a = s_or[s_or.escenario == "principal"].set_index("marcador")
b = s_or[s_or.escenario == "deduplicado"].set_index("marcador")
dentro = int(((b["OR"] >= a["ci_lo"]) & (b["OR"] <= a["ci_hi"])).sum())
print(f"max |Δ| prevalence {max_dif(s_prev, 'entidad', 'prevalencia'):.2f} pp · "
      f"burden {max_dif(s_prev, 'entidad', 'burden'):.2f} pp · "
      f"burden by facility type {max_dif(s_tipo, ['tipo', 'entidad'], 'burden'):.2f} pp · "
      f"OR {max_dif(s_or, 'marcador', 'OR'):.3f}")
print(f"deduplicated ORs inside the main 95% CI: {dentro}/{len(a)}")

## 7 · Table 1 and Figures 1–2

**Negative review burden** = P(review mentions the entity ∧ review is negatively rated) = prevalence × negatively rated share / 100. Being a binomial proportion, it admits a 95% CI by **bootstrap clustered by facility** (resampling facilities, not reviews).

In [ ]:
RNG4 = np.random.default_rng(SEED)        # shared stream: Table 1 first, then burden by facility type (order matters)


def boot_burden(n_rev, n_hit, rng=RNG4, n_rep=N_BOOT):
    k = len(n_rev)
    out = np.empty(n_rep)
    for i in range(n_rep):
        ix = rng.integers(0, k, k)
        tot = n_rev[ix].sum()
        out[i] = n_hit[ix].sum() / tot * 100 if tot else np.nan
    return np.nanpercentile(out, 2.5), np.nanpercentile(out, 97.5)


ORDEN = ENTIDADES_FIABLES + ["Instalaciones"]
filas = []
for e in ORDEN:
    m = ENT[e]
    fila = {"entity": ETIQ_EN[e], "validated": "no (exploratory)" if e == "Instalaciones" else "yes",
            "mentions_n": int(m.sum())}
    for et, sel in [("overall", np.ones(N, bool)), ("primary", PRIM), ("hospital", HOSP)]:
        g = pd.DataFrame({"c": CEN[sel], "m": m[sel], "mn": (m & NEG)[sel]}).groupby("c").agg(
            n=("m", "size"), hit=("mn", "sum"))
        lo, hi = boot_burden(g["n"].to_numpy(), g["hit"].to_numpy())
        fila[f"prev_{et}"] = m[sel].mean() * 100
        fila[f"negrate_{et}"] = NEG[sel][m[sel]].mean() * 100
        fila[f"burden_{et}"] = g["hit"].sum() / g["n"].sum() * 100
        fila[f"burden_{et}_lo"], fila[f"burden_{et}_hi"] = lo, hi
    filas.append(fila)
t1 = pd.DataFrame(filas)
t1.to_csv(DIRS["tables"] / "Table_1_entity_prevalence_negativity_burden.csv", index=False)
t1[["entity", "validated", "mentions_n", "prev_overall", "negrate_overall", "burden_overall",
    "burden_overall_lo", "burden_overall_hi", "burden_primary", "burden_hospital"]].round(1)

In [ ]:
# Figure 1 · salience × negatively rated share, with isolines of constant burden (x·y = k)
ANCLA_FIG1 = {   # (point of the pair, dx, dy, alignment): each label anchored to one point
    "Sistema/Gestión": ("prim", 10, -2, "left"), "Médico": ("prim", 10, 2, "left"),
    "Administrativos": ("prim", -9, 5, "right"), "Urgencias": ("hosp", 9, 2, "left"),
    "Enfermería": ("hosp", 9, -1, "left"), "Instalaciones": ("hosp", 9, -8, "left"),
}
fig, ax = plt.subplots(figsize=(7.2, 5.4))
xmax = max(max(ENT[e][PRIM].mean() for e in ORDEN) * 100 * 1.25, 60)
xs = np.linspace(0.5, xmax, 400)
for k in [5, 10, 20, 30, 40, 50]:
    ys = k * 100 / xs
    ax.plot(xs[ys <= 100], ys[ys <= 100], color="#bbbbbb", lw=0.7, ls=":", zorder=1)
    if k * 100 / 97 < xmax:
        ax.text(k * 100 / 97, 97.5, f"{k}", color="#999999", fontsize=7, ha="center", va="bottom")
for e in ORDEN:
    m = ENT[e]
    pts = [(m[s].mean() * 100, NEG[s][m[s]].mean() * 100, int(m[s].sum())) for s in (PRIM, HOSP)]
    ax.plot([pts[0][0], pts[1][0]], [pts[0][1], pts[1][1]], color=COL["gris"], lw=0.8, alpha=0.55, zorder=2)
    expl = e == "Instalaciones"
    for (x, y, n), lvl in zip(pts, ["Primaria", "Hospitalaria"]):
        ax.scatter(x, y, s=28 + n / 120, color=COL[lvl], zorder=3,
                   marker="s" if lvl == "Primaria" else "o", alpha=0.45 if expl else 1.0,
                   edgecolor="white", linewidth=0.6)
    anc, dx, dy, ha = ANCLA_FIG1[e]
    x0, y0 = pts[0][:2] if anc == "prim" else pts[1][:2]
    ax.annotate(ETIQ_EN[e] + (" *" if expl else ""), (x0, y0), textcoords="offset points",
                xytext=(dx, dy), fontsize=8.5, ha=ha, color="#888888" if expl else "#222222",
                style="italic" if expl else "normal")
ax.set_xlabel("Mention frequency (% of reviews)")
ax.set_ylabel("Negatively rated share of mentions (%)")
ax.set_ylim(0, 102); ax.set_xlim(0, xmax)
ax.set_title("Entity salience and negatively rated share by level of care\n"
             "dotted curves = constant negative review burden", fontsize=10, loc="left")
ax.legend(handles=[Line2D([], [], marker="s", ls="", color=COL["Primaria"], label="Primary care"),
                   Line2D([], [], marker="o", ls="", color=COL["Hospitalaria"], label="Hospital care"),
                   Line2D([], [], ls=":", color="#bbbbbb", label="burden isoline"),
                   Line2D([], [], marker="s", ls="", color="#bbbbbb",
                          label="* exploratory (failed validation)")],
          loc="lower right", frameon=False, fontsize=8)
fig.tight_layout()
fig.savefig(DIRS["figures"] / "Figure_1_entity_salience_and_negatively_rated_share.png",
            dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Figure 2 · burden by facility type × validated entity (Facilities excluded)
mat = {ETIQ_EN[e]: pd.Series(ENT[e] & NEG, index=corpus.index).groupby(corpus["tipo"]).mean() * 100
       for e in ENTIDADES_FIABLES}
m2 = pd.DataFrame(mat).sort_values("System/Management", ascending=False)
n_tipo = corpus.groupby("tipo").size()

fig, ax = plt.subplots(figsize=(7.6, 4.6))
im = ax.imshow(m2.values, cmap="YlOrRd", aspect="auto", vmin=0, vmax=m2.values.max())
ax.set_xticks(range(m2.shape[1])); ax.set_xticklabels(m2.columns, rotation=20, ha="right")
ax.set_yticks(range(m2.shape[0]))
ax.set_yticklabels([f"{i}  (n={n_tipo[i]:,})" for i in m2.index], fontsize=8)
for i in range(m2.shape[0]):
    for j in range(m2.shape[1]):
        v = m2.values[i, j]
        ax.text(j, i, f"{v:.1f}", ha="center", va="center", fontsize=8,
                color="white" if v > m2.values.max() * 0.6 else "#222222")
ax.set_title("Negative review burden by facility type (validated entities)\n"
             "rows ordered by System/Management burden", fontsize=10, loc="left")
ax.grid(False)
fig.colorbar(im, ax=ax, shrink=0.8, label="burden (% of all reviews)")
fig.tight_layout()
fig.savefig(DIRS["figures"] / "Figure_2_negative_review_burden_by_facility_type.png",
            dpi=300, bbox_inches="tight")
plt.show()
m2.round(1)

In [ ]:
# Burden by facility type × entity with facility-clustered 95% CI, and the seven outpatient
# consultation centres (smallest type, highest burden). Written to Multimedia Appendix 3 in §9.
sup = []
for tipo in m2.index:
    sel = (corpus["tipo"] == tipo).to_numpy()
    for e in ENTIDADES_FIABLES:
        hit = (ENT[e] & NEG)[sel]
        g = pd.DataFrame({"c": CEN[sel], "h": hit}).groupby("c").agg(n=("h", "size"), hit=("h", "sum"))
        lo, hi = boot_burden(g["n"].to_numpy(), g["hit"].to_numpy())
        sup.append({"facility_type": tipo, "entity": ETIQ_EN[e], "n_reviews": int(sel.sum()),
                    "n_facilities": int(pd.Series(CEN[sel]).nunique()),
                    "burden": round(hit.mean() * 100, 2), "ci_lo": round(lo, 2), "ci_hi": round(hi, 2)})
burden_tipo = pd.DataFrame(sup)

oc = corpus[corpus["tipo"] == "Outpatient consultation centre"]
h = pd.Series(ENT["Sistema/Gestión"] & NEG, index=corpus.index).loc[oc.index]
outpatient = (oc.assign(h=h).groupby("Nombre")
              .agg(province=("Ciudad", "first"), n_reviews=("h", "size"), n_hit=("h", "sum")))
outpatient["burden_pct"] = (outpatient["n_hit"] / outpatient["n_reviews"] * 100).round(1)
outpatient = outpatient.sort_values("n_reviews", ascending=False).reset_index()
outpatient_summary = pd.DataFrame(
    [("Range of System/Management burden across the 7 outpatient consultation centres",
      f"{outpatient.burden_pct.min():.1f}% to {outpatient.burden_pct.max():.1f}%"),
     ("Provincial distribution", ", ".join(f"{k}: {v}" for k, v in outpatient.province.value_counts().items())),
     ("Largest centre share of the type's reviews",
      f"{outpatient.n_reviews.max() / outpatient.n_reviews.sum() * 100:.0f}%")],
    columns=["item", "value"])
burden_tipo[burden_tipo.entity == "System/Management"]

## 8 · Table 2 and n/N dossier

Lexical markers are compared between levels **within the sentiment stratum** (negative markers among negative reviews, positive markers among positive reviews), by logistic regression with SE clustered by facility. They are high-precision, low-recall markers of stereotyped formulations, not estimators of the underlying concept.

In [ ]:
# ── Table 2 ──
dfm = corpus[corpus["Atencion"].isin(["Primaria", "Hospitalaria"])].copy()
dfm["Atencion"], dfm["Nombre"] = dfm["Atencion"].astype(str), dfm["Nombre"].astype(str)
t2 = []
for c in CON_V2:
    est = "positivo" if c in MARCADORES_POSITIVOS else "negativo"
    dfm["y"] = CON[c][dfm.index.to_numpy()].astype(int)
    s = dfm[dfm["sentimiento"] == est]
    OR, lo, hi, p = logit_or(s, "y")
    sp, sh = s[s.Atencion == "Primaria"], s[s.Atencion == "Hospitalaria"]
    t2.append({"marker": ETIQ_M[c], "stratum": "negative reviews" if est == "negativo" else "positive reviews",
               "n_stratum": len(s), "n_primary": int(sp["y"].sum()), "N_primary": len(sp),
               "prev_primary_pct": sp["y"].mean() * 100, "n_hospital": int(sh["y"].sum()),
               "N_hospital": len(sh), "prev_hospital_pct": sh["y"].mean() * 100,
               "OR": OR, "ci_lo": lo, "ci_hi": hi, "p": p})
t2 = pd.DataFrame(t2)
t2.to_csv(DIRS["tables"] / "Table_2_lexical_markers.csv", index=False)
t2.round(3)

In [ ]:
# ── n/N dossier: numerator and denominator of every percentage quoted in Results ──
print(f"coverage (>=1 entity) {cob.mean()*100:5.1f}% ({cob.sum():,}/{N:,})\n")
for e in ORDEN:
    m = ENT[e]
    a, b = m & PRIM, m & HOSP
    print(f"{ETIQ_EN[e]:<22} prev {m.mean()*100:5.1f}% ({m.sum():,}/{N:,}) | "
          f"neg {NEG[m].mean()*100:5.1f}% ({int((m & NEG).sum()):,}/{m.sum():,}) | "
          f"burden {(m & NEG).mean()*100:5.1f}% ({int((m & NEG).sum()):,}/{N:,})")
    print(f"{'':<22} prev primary {m[PRIM].mean()*100:5.1f}% ({int(a.sum()):,}/{PRIM.sum():,}) | "
          f"hospital {m[HOSP].mean()*100:5.1f}% ({int(b.sum()):,}/{HOSP.sum():,}) | "
          f"neg primary {NEG[a].mean()*100:5.1f}% ({int((a & NEG).sum()):,}/{a.sum():,}) | "
          f"hospital {NEG[b].mean()*100:5.1f}% ({int((b & NEG).sum()):,}/{b.sum():,})")
print()
for r in t2.itertuples():
    print(f"{r.marker:<22} primary {r.prev_primary_pct:5.1f}% ({r.n_primary:,}/{r.N_primary:,}) | "
          f"hospital {r.prev_hospital_pct:5.1f}% ({r.n_hospital:,}/{r.N_hospital:,})")

## 9 · Multimedia Appendices 1–4

| Appendix | File | Content |
|---|---|---|
| 1 | `MultimediaAppendix1_lexicon_dictionaries.xlsx` | Entity and lexical-marker dictionaries of the final lexicon, phrase lists, disambiguation rules |
| 2 | `MultimediaAppendix2_annotation_guide.csv` | Manual annotation guide used for validation, as delivered to the annotators (Spanish) |
| 3 | `MultimediaAppendix3_burden_by_facility_type.xlsx` | Burden by facility type with 95% CIs, facility classification mapping, outpatient consultation centres |
| 4 | `MultimediaAppendix4_sensitivity_analyses.xlsx` | Sensitivity analyses S1–S4 |

Multimedia Appendix 5 (validation details) needs the restricted annotation and is written in section 10.

In [ ]:
MA = DIRS["multimedia_appendices"]

# ── Multimedia Appendix 1 · entity and lexical-marker dictionaries ──
ent_lex = []
for e, terms in ENT_V2_SUBSTR.items():
    ent_lex += [{"entity": ETIQ_EN[e], "matching": "surface substring", "pattern": x} for x in terms]
    ent_lex += [{"entity": ETIQ_EN[e], "matching": "regular expression", "pattern": x}
                for x in ENT_V2_REGEX.get(e, [])]
ent_lex.append({"entity": "Administrative staff", "matching": "contextual window",
                "pattern": "información within +/-3 tokens of: " + ", ".join(sorted(INFO_ANCLAS))})
reglas = pd.DataFrame([
    ("due", "substring", r"\bdue\b", "full word-boundary matching; removes duele, dueño, duermen"),
    ("lista", "substring", r"\blistas? de espera\b", "full phrase required; removes especialista"),
    ("cita", "substring", r"\bcita", "initial word-boundary anchoring; removes felicitar, felicito"),
    ("sala", "substring", r"\bsalas?\b(?!\s+(?:de\s+)?espera)", "excluded when followed by espera or de espera"),
    ("información", "substring", "contextual window +/-3 tokens", "administrative anchor required"),
    ("personal sanitario", "phrase", "(removed)", "dropped from professionalism marker"),
    ("atención recibida", "phrase", "(removed)", "dropped from gratitude marker"),
], columns=["trigger", "original_rule", "final_rule", "rationale"])
frases = pd.DataFrame([{"marker": ETIQ_M[c], "phrase": fr,
                        "n_reviews_matched": int(T.str.contains(re.escape(fr), na=False).sum()),
                        "in_original_lexicon": fr in CON_ORIG[c]}
                       for c, lista in CON_V2.items() for fr in lista])
notas1 = pd.DataFrame([
    ("Final lexicon", "The lexicon used for every estimate in the paper. It was fixed before manual "
                      "validation and not subsequently modified."),
    ("Original lexicon", "The lexicon before the disambiguation rules were applied. Used only in "
                         "sensitivity analysis S2 (Multimedia Appendix 4)."),
    ("n_reviews_matched", "Number of reviews in the corpus containing this phrase. NOT additive: a "
                          "single review may contain several phrases of the same marker, so the "
                          "column does not sum to the number of reviews assigned to the marker."),
    ("Phrases with zero matches", "Retained because they belong to the final lexicon as fixed before "
                                  "manual validation; they were part of the lexicon as applied and are "
                                  "reported for completeness."),
    ("in_original_lexicon", "FALSE identifies phrases introduced in the final lexicon by the "
                            "reconstruction of the waiting marker, before manual validation."),
], columns=["field", "definition"])
f = MA / "MultimediaAppendix1_lexicon_dictionaries.xlsx"
with pd.ExcelWriter(f, engine="openpyxl") as x:
    pd.DataFrame(ent_lex).to_excel(x, sheet_name="entity lexicon", index=False)
    reglas.to_excel(x, sheet_name="disambiguation rules", index=False)
    frases.to_excel(x, sheet_name="marker phrases", index=False)
    pd.DataFrame([{"entity": ETIQ_EN[k], "n_terms": len(v)} for k, v in ENT_ORIG.items()]).to_excel(
        x, sheet_name="original lexicon term counts", index=False)
    notas1.to_excel(x, sheet_name="notes", index=False)
formatear(f, anchos={"entity": 22, "matching": 20, "pattern": 58, "trigger": 20, "original_rule": 16,
                     "final_rule": 40, "rationale": 56, "marker": 24, "phrase": 34,
                     "definition": 90, "field": 26},
          wrap_cols={"pattern", "rationale", "final_rule", "definition"})

# ── Multimedia Appendix 2 · annotation guide ──
pd.DataFrame(CODEBOOK, columns=["field", "definition"]).to_csv(
    MA / "MultimediaAppendix2_annotation_guide.csv", index=False)

In [ ]:
# ── Multimedia Appendix 3 · burden by facility type, classification mapping, outpatient centres ──
g5 = corpus.groupby("Tipo_de_centro").agg(n_reviews=("Nombre", "size"), n_facilities=("Nombre", "nunique"),
                                          level=("Atencion", "first"))
mapping = pd.DataFrame([{"Tipo_de_centro (registry field, Spanish)": k, "Analytic category (English)": v,
                         "n_reviews": int(g5.loc[k, "n_reviews"]), "n_facilities": int(g5.loc[k, "n_facilities"]),
                         "level_of_care": {"Primaria": "Primary care", "Hospitalaria": "Hospital care"}[g5.loc[k, "level"]]}
                        for k, v in MAPA_TIPO.items()])
notas3 = pd.DataFrame([
    ("burden", "Negative review burden: % of all reviews of the facility type that mention the entity "
               "and are negatively rated. 95% CI by bootstrap resampling facilities (2,000 replicates)."),
    ("Facility classification", "The registry field contains two spellings of the health centre "
                                "category (Centro de Salud / Centro de salud); both map to the same "
                                "analytic category, which is why nine analytic categories arise from "
                                "ten registry values."),
    ("Outpatient consultation centres", "All seven centres are located in Seville province, so the "
                                        "facility-type estimate cannot be disentangled from "
                                        "province-specific context."),
], columns=["field", "definition"])
f = MA / "MultimediaAppendix3_burden_by_facility_type.xlsx"
with pd.ExcelWriter(f, engine="openpyxl") as x:
    burden_tipo.to_excel(x, sheet_name="burden by type and entity", index=False)
    mapping.to_excel(x, sheet_name="facility classification", index=False)
    outpatient.to_excel(x, sheet_name="outpatient centres detail", index=False)
    outpatient_summary.to_excel(x, sheet_name="outpatient centres summary", index=False)
    notas3.to_excel(x, sheet_name="notes", index=False)
formatear(f, anchos={"facility_type": 30, "entity": 22, "Nombre": 62, "item": 60, "value": 40,
                     "Tipo_de_centro (registry field, Spanish)": 42, "Analytic category (English)": 34,
                     "field": 30, "definition": 85},
          wrap_cols={"definition"})
mapping

In [ ]:
# ── Multimedia Appendix 4 · sensitivity analyses S1-S4 ──
def resumen(d, ent_d, con_d):
    neg = d["negativa"].to_numpy().astype(bool)
    prev = {ETIQ_EN[e]: np.asarray(ent_d[e], bool).mean() * 100 for e in ETIQ_EN}
    bur = {ETIQ_EN[e]: (np.asarray(ent_d[e], bool) & neg).mean() * 100 for e in ETIQ_EN}
    dm = d[d["Atencion"].isin(["Primaria", "Hospitalaria"])].copy()
    dm["Atencion"], dm["Nombre"] = dm["Atencion"].astype(str), dm["Nombre"].astype(str)
    ors = {}
    for c in CON_V2:
        dm["y"] = np.asarray(con_d[c])[dm.index.to_numpy()].astype(int)
        est = "positivo" if c in MARCADORES_POSITIVOS else "negativo"
        ors[ETIQ_M[c]] = logit_or(dm[dm["sentimiento"] == est], "y")[0]
    return prev, bur, ors


def escenario(d):
    d = d.reset_index(drop=True)
    return d, detectar_entidades(d["text_clean"], "v2"), detectar_constructos(d["text_clean"], "v2")


base = resumen(corpus, ENT, CON)
alternativas = {
    "S1 restricted to 2016-2025": escenario(corpus[corpus["year"].between(2016, 2025)]),
    "S2 original lexicon": (corpus, ENT_O, CON_O),
    "S3 deduplicated": escenario(corpus[~DUP]),
}
sens = []
for nombre, (d, e_, c_) in alternativas.items():
    alt = resumen(d, e_, c_)
    for e in ETIQ_EN:
        for q, i in [("prevalence", 0), ("burden", 1)]:
            sens.append({"analysis": nombre, "quantity": q, "item": ETIQ_EN[e],
                         "main_estimate": base[i][ETIQ_EN[e]], "sensitivity_estimate": alt[i][ETIQ_EN[e]]})
    for m_ in alt[2]:
        sens.append({"analysis": nombre, "quantity": "odds ratio", "item": m_,
                     "main_estimate": base[2][m_], "sensitivity_estimate": alt[2][m_]})
sens = pd.DataFrame(sens)
sens["abs_change"] = (sens["sensitivity_estimate"] - sens["main_estimate"]).abs()
sens[["main_estimate", "sensitivity_estimate", "abs_change"]] = \
    sens[["main_estimate", "sensitivity_estimate", "abs_change"]].round(3)
max_cambio = (sens.groupby(["analysis", "quantity"])["abs_change"].max().reset_index()
              .rename(columns={"abs_change": "max_absolute_change"}))

# S3 at facility-type level
dd = corpus[~DUP].reset_index(drop=True)
ent_dd = detectar_entidades(dd["text_clean"], "v2")
neg_dd = dd["negativa"].to_numpy().astype(bool)
s3t = []
for tipo in corpus["tipo"].dropna().unique():
    m1, m2_ = (corpus["tipo"] == tipo).to_numpy(), (dd["tipo"] == tipo).to_numpy()
    for e in ENTIDADES_FIABLES:
        b1 = (ENT[e] & NEG)[m1].mean() * 100
        b2 = (np.asarray(ent_dd[e], bool) & neg_dd)[m2_].mean() * 100
        s3t.append({"facility_type": tipo, "entity": ETIQ_EN[e], "n_reviews_main": int(m1.sum()),
                    "n_reviews_dedup": int(m2_.sum()), "main_burden_pct": round(b1, 3),
                    "deduplicated_burden_pct": round(b2, 3), "absolute_change_pp": round(abs(b2 - b1), 3)})
s3t = pd.DataFrame(s3t).sort_values(["entity", "main_burden_pct"], ascending=[True, False]).reset_index(drop=True)

# S4 period stratification (2016-2025, both levels)
c4 = corpus[corpus["Atencion"].isin(["Primaria", "Hospitalaria"]) & corpus["year"].between(2016, 2025)].copy()
c4["period"] = np.select([c4["year"] <= 2019, c4["year"] <= 2021],
                         ["pre-pandemic (2016-2019)", "pandemic (2020-2021)"], default="post-pandemic (2022-2025)")
c4["level"] = c4["Atencion"].map({"Primaria": "Primary care", "Hospitalaria": "Hospital care"})
s4 = []
for e in ENTIDADES_FIABLES + ["Instalaciones"]:
    m = ENT[e][c4.index.to_numpy()]
    g = c4.assign(m=m, mn=m & c4["negativa"].to_numpy().astype(bool))
    for (lvl, per), s in g.groupby(["level", "period"]):
        s4.append({"entity": ETIQ_EN[e], "level": lvl, "period": per, "n_reviews": len(s),
                   "n_mentions": int(s["m"].sum()), "prevalence_pct": round(s["m"].mean() * 100, 2),
                   "pct_negative_of_mentions": round(s.loc[s["m"], "negativa"].mean() * 100, 2),
                   "burden_pct": round(s["mn"].mean() * 100, 2)})
s4 = pd.DataFrame(s4)
notas4 = pd.DataFrame([
    ("Period boundaries", "pre-pandemic = 2016-2019; pandemic = 2020-2021; post-pandemic = 2022-2025"),
    ("Corpus", "restricted to 2016-2025 and to the two levels of care"),
    ("prevalence_pct", "n_mentions / n_reviews within the level-period cell"),
    ("pct_negative_of_mentions", "share of mentioning reviews that are negatively rated"),
    ("burden_pct", "share of all reviews in the cell that both mention the entity and are negatively rated"),
    ("Facilities", "shown for completeness; did not pass validation and is not interpreted"),
], columns=["field", "definition"])

f = MA / "MultimediaAppendix4_sensitivity_analyses.xlsx"
with pd.ExcelWriter(f, engine="openpyxl") as x:
    sens.to_excel(x, sheet_name="S1-S3 estimates", index=False)
    s3t.to_excel(x, sheet_name="S3 facility-type burden", index=False)
    s4.to_excel(x, sheet_name="S4 period stratification", index=False)
    notas4.to_excel(x, sheet_name="S4 notes", index=False)
    max_cambio.to_excel(x, sheet_name="max absolute change", index=False)
formatear(f, anchos={"analysis": 26, "item": 24, "quantity": 13, "facility_type": 30, "entity": 22,
                     "period": 26, "level": 14, "definition": 80, "field": 26}, wrap_cols={"definition"})
max_cambio

## 10 · Table 3 and Multimedia Appendix 5 *(needs section 5)*

When every sampled lexicon-positive case is a true positive, the non-parametric bootstrap cannot generate a false positive and the precision interval collapses to a point (1.000–1.000). In those cases the reported interval is a **Wilson interval on the raw counts**; the `*_ci_method` columns record which was used.

In [ ]:
f_val = DIRS["validation"] / "validation_metrics.csv"
if f_val.exists():
    v = pd.read_csv(f_val)
    v_est = pd.read_csv(DIRS["validation"] / "validation_precision_by_stratum.csv")
    v_k = pd.read_csv(DIRS["validation"] / "validation_interrater_agreement.csv")

    def ic_precision(lo, hi, tp, n):
        """Bootstrap CI, or Wilson on raw counts when the bootstrap degenerates."""
        if abs(hi - lo) < 1e-9 and n > 0:
            w = wilson(tp, n)
            return round(w[0], 3), round(w[1], 3), f"Wilson on {tp}/{n} (bootstrap degenerate)"
        return round(lo, 3), round(hi, 3), "bootstrap"

    # ── Table 3 ──
    t3 = v.merge(v_k[["variable", "kappa", "acuerdo_bruto"]], left_on="diana",
                 right_on="variable", how="left")
    t3.insert(0, "tipo", np.where(t3.diana.str.startswith("ent_"), "entity", "lexical marker"))
    ic = [ic_precision(r.AB_prec_lo, r.AB_prec_hi, r.AB_tp, r.AB_n_pred_pos) for r in t3.itertuples()]
    t3["AB_prec_lo"], t3["AB_prec_hi"], t3["AB_precision_ci_method"] = zip(*ic)
    t3 = t3[["tipo", "diana", "n_marcadas_corpus", "Aps_precision", "Aps_prec_lo", "Aps_prec_hi",
             "Aps_recall", "Aps_rec_lo", "Aps_rec_hi", "AB_precision", "AB_prec_lo", "AB_prec_hi",
             "AB_recall", "AB_rec_lo", "AB_rec_hi", "AB_f1", "kappa", "acuerdo_bruto",
             "AB_precision_ci_method"]]
    t3.to_csv(DIRS["tables"] / "Table_3_validation_metrics.csv", index=False)

    # ── Multimedia Appendix 5 · the three estimators side by side + trigger-level audit ──
    val_det = pd.DataFrame({
        "target": v.diana, "n_flagged_corpus": v.n_marcadas_corpus.astype(int),
        "A_crude_precision": v.A_crudo_precision.round(3), "A_crude_recall": v.A_crudo_recall.round(3),
        "A_poststratified_precision": v.Aps_precision.round(3),
        "A_poststratified_prec_lo": v.Aps_prec_lo, "A_poststratified_prec_hi": v.Aps_prec_hi,
        "A_poststratified_recall": v.Aps_recall.round(3),
        "A_poststratified_rec_lo": v.Aps_rec_lo.round(3), "A_poststratified_rec_hi": v.Aps_rec_hi.round(3),
        "AB_calibrated_precision": v.AB_precision.round(3),
        "AB_calibrated_prec_lo": v.AB_prec_lo, "AB_calibrated_prec_hi": v.AB_prec_hi,
        "AB_calibrated_recall": v.AB_recall.round(3),
        "AB_calibrated_rec_lo": v.AB_rec_lo.round(3), "AB_calibrated_rec_hi": v.AB_rec_hi.round(3),
        "AB_calibrated_F1": v.AB_f1.round(3)})
    ic_ab = [ic_precision(round(r.AB_prec_lo, 3), round(r.AB_prec_hi, 3), r.AB_tp, r.AB_n_pred_pos)
             for r in v.itertuples()]
    ic_a = [ic_precision(round(r.Aps_prec_lo, 3), round(r.Aps_prec_hi, 3), r.A_tp, r.A_n_pred_pos)
            for r in v.itertuples()]
    val_det["AB_calibrated_prec_lo"], val_det["AB_calibrated_prec_hi"], val_det["AB_precision_ci_method"] = zip(*ic_ab)
    val_det["A_poststratified_prec_lo"], val_det["A_poststratified_prec_hi"], \
        val_det["A_poststratified_precision_ci_method"] = zip(*ic_a)

    trig = v_est.rename(columns={"estrato": "sampling_stratum", "diana": "target",
                                 "n_anotadas": "n_annotated", "n_correctas": "n_correct",
                                 "poblacion_estrato": "stratum_population_in_corpus"})
    trig[["precision", "ci_lo", "ci_hi"]] = trig[["precision", "ci_lo", "ci_hi"]].round(3)
    notas7 = pd.DataFrame([
        ("A crude", "Frame A unweighted. Auditable by hand but NOT an estimator of corpus-level "
                    "performance: Frame A was sampled 100 primary + 100 hospital (50:50), which is "
                    "not the corpus composition (60.3% / 39.7%)."),
        ("A post-stratified", "Frame A reweighted to the true level proportions. Frame A estimator "
                              "of corpus-level performance. Bootstrap intervals by level."),
        ("A+B calibrated", "Frames A and B with design weights calibrated to census cell totals. "
                           "Uses the full validation sample and is used as the primary estimator. "
                           "Stratified bootstrap recalibrating within each replicate."),
        ("Degenerate intervals", "Where every sampled lexicon-positive case was a true positive, the "
                                 "non-parametric bootstrap cannot generate a false positive and the "
                                 "interval collapses to a point. In those cases the reported interval "
                                 "is a Wilson interval on the raw counts; the column "
                                 "*_ci_method records which was used. The raw bootstrap result was "
                                 "(1.000-1.000)."),
        ("Agreement across estimators", "Close agreement indicates that a conclusion does not depend "
                                        "on the weighting scheme."),
        ("Trigger-level audit", "Precision within targeted Frame B strata, unweighted, Wilson intervals. "
                                "These strata isolate reviews whose entity assignment depended on a "
                                "single trigger."),
    ], columns=["item", "definition"])
    f = DIRS["multimedia_appendices"] / "MultimediaAppendix5_validation_details.xlsx"
    with pd.ExcelWriter(f, engine="openpyxl") as x:
        val_det.to_excel(x, sheet_name="three estimators", index=False)
        trig.to_excel(x, sheet_name="trigger-level audit", index=False)
        notas7.to_excel(x, sheet_name="notes", index=False)
    formatear(f, wrap_cols={"definition"})
    display(t3.round(3))
else:
    print("Section 10 skipped: run section 5 first (needs the restricted annotation files).")